# Shadow training — Kaggle GPU (parallel)

**One notebook trains one variant.** Set `VARIANT` in the first cell, then run top-to-bottom. Duplicate this notebook 3 times (one per variant) and run all three in parallel — Kaggle allows up to 4 concurrent GPU sessions per account. Total wall time ≈ 4 hours.

**Variants:**
- `rgb` — train on raw images (RGB baseline)
- `shadow_vanilla` — train on Shadow-edited images (paper's method)
- `shadow_noise` — train on Shadow-edited images with calibration noise injection (our extension)

**Setup checklist (one time):**
1. Create a Kaggle account, verify phone (required for GPU).
2. Push your shadow project to GitHub (it's ok if it's private — Kaggle can clone it with a token).
3. Set this notebook's *Settings → Accelerator → GPU P100*, and *Internet → On*.

In [ ]:
# ============== EDIT THIS ==============
VARIANT      = 'rgb'                     # rgb | shadow_vanilla | shadow_noise
SEED         = 0
NUM_EPOCHS   = 2000                      # paper used 5000; 2000 converges fine for Stack
GITHUB_REPO  = 'https://github.com/<YOUR_USERNAME>/shadow.git'  # public or token-auth URL
# =======================================

assert VARIANT in {'rgb', 'shadow_vanilla', 'shadow_noise'}
print(f'will train {VARIANT} (seed {SEED}) for {NUM_EPOCHS} epochs')

## 1. Install dependencies (~5 min)

Pull robosuite v1.5.1, robomimic v0.5.0, mimicgen 1.0, and apply the three patches we documented in `CLAUDE.md`.

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d robosuite ] || git clone -q https://github.com/ARISE-Initiative/robosuite.git
[ -d robomimic ] || git clone -q https://github.com/ARISE-Initiative/robomimic.git
[ -d mimicgen  ] || git clone -q https://github.com/NVlabs/mimicgen.git
(cd robosuite && git checkout -q v1.5.1)

# --- patch 1: egl_probe is Linux-multi-GPU-only; harmless to keep on Kaggle P100 but easier to drop
sed -i 's/.*egl_probe.*$//' robomimic/setup.py
python - <<'PY'
p = 'robomimic/robomimic/envs/env_robosuite.py'
s = open(p).read()
if 'try:\n                    import egl_probe' not in s:
    s = s.replace(
        'import egl_probe\n                valid_gpu_devices = egl_probe.get_available_devices()',
        'try:\n                    import egl_probe\n                    valid_gpu_devices = egl_probe.get_available_devices()\n                except ImportError:\n                    valid_gpu_devices = []')
open(p, 'w').write(s)
PY

# --- patch 2: SingleArmEnv compat shim for mimicgen
cat > robosuite/robosuite/environments/manipulation/single_arm_env.py <<'PY'
from robosuite.environments.manipulation.manipulation_env import ManipulationEnv as SingleArmEnv  # noqa: F401
PY
echo 'patches applied'

In [ ]:
%%bash
set -e
cd /kaggle/working
pip install -q 'mujoco==3.2.7' cmake 'numpy<2.3' scipy seaborn
pip install -q -e ./robosuite
pip install -q -e ./robomimic
pip install -q -e ./mimicgen
python robosuite/robosuite/scripts/setup_macros.py >/dev/null 2>&1 || true

# diffusers' Flax schedulers break on newer JAX. We use the PyTorch backend only.
# Comment out every Flax import in diffusers/schedulers/__init__.py so `import
# diffusers` stops crashing. Done in Python (not sed) so escaping is unambiguous.
python <<'PYEOF'
import sysconfig, os, re
init = os.path.join(sysconfig.get_paths()["purelib"], "diffusers", "schedulers", "__init__.py")
if not os.path.exists(init):
    print("NOT FOUND:", init); raise SystemExit(0)
src = open(init).read()
patched = re.sub(r"^(from \.scheduling_\w*flax\w*.*)$",
                 r"# DISABLED (Flax not used; breaks on newer JAX): \1",
                 src, flags=re.M)
if patched != src:
    open(init, "w").write(patched)
    print("patched", init)
else:
    print("no flax lines found (already patched?)", init)
PYEOF
echo 'install done'


## 2. Clone the shadow project from GitHub

In [ ]:
%%bash -s "$GITHUB_REPO"
set -e
cd /kaggle/working
[ -d shadow ] || git clone -q "$1" shadow
cd shadow && git pull -q
ls

## 3. Build the dataset for this variant (~30–60 min)

RGB needs nothing — it uses the raw Mimicgen Stack dataset. Both Shadow variants run the IK+render+composite pipeline.

In [ ]:
%%bash -s "$VARIANT"
set -e
cd /kaggle/working/shadow
mkdir -p datasets/core datasets/shadow
python /kaggle/working/mimicgen/mimicgen/scripts/download_datasets.py --download_dir datasets --dataset_type core --tasks stack_d0

case "$1" in
  rgb) echo 'RGB uses raw dataset, nothing to build' ;;
  shadow_vanilla) python create_shadow_dataset.py --src datasets/core/stack_d0.hdf5 --out datasets/shadow/stack_d0_shadow.hdf5       --source_robot Panda --target_robot Sawyer --noise none ;;
  shadow_noise)   python create_shadow_dataset.py --src datasets/core/stack_d0.hdf5 --out datasets/shadow/stack_d0_shadow_noise.hdf5 --source_robot Panda --target_robot Sawyer --noise gaussian --sigma_xyz 0.01 --sigma_rot_deg 5.0 ;;
esac

## 4. Train (~2.5–3.5h on Kaggle P100)

In [ ]:
import subprocess
subprocess.run(
    ['python', 'train.py',
     '--variant', VARIANT,
     '--seed', str(SEED),
     '--epochs', str(NUM_EPOCHS)],
    cwd='/kaggle/working/shadow',
    check=True,
)

## 5. Save the checkpoint to Kaggle's output (so you can download it)

In [ ]:
%%bash -s "$VARIANT" "$SEED"
set -e
RUN_DIR=$(ls -td /kaggle/working/shadow/results/runs/* 2>/dev/null | head -1)
BEST_CKPT=$(find "$RUN_DIR" -name 'model_epoch_*.pth' | sort -V | tail -1)
echo "saving $BEST_CKPT"
mkdir -p /kaggle/working/checkpoints
cp "$BEST_CKPT" "/kaggle/working/checkpoints/${1}_seed${2}.pth"
ls -lh /kaggle/working/checkpoints/